In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import popsim.param_utils as param_utils
from popsim.modules.tearing import DisruptionPhase, IslandRotationPhase, Tearing
from popsim.simulate import simulate

dt = 1e-4  # s
time_base = param_utils.make_time_base(t0=0.0, t1=7.0, dt=dt)
config = Tearing.Config(magx_time=time_base, thincurr_file="21_mode_resp_data.txt", ods_file="thatfile.txt")


def generate_disruption_phase_trajectory(trigger_time: float, tq_to_cq_dur: float):
    # TODO(allenw): we want a rectilinear interpolation scheme.
    disrupt_phase_dict = {
        0.0: DisruptionPhase.NONE,
        trigger_time - dt: DisruptionPhase.NONE,
        trigger_time: DisruptionPhase.TQ,
        trigger_time + tq_to_cq_dur - dt: DisruptionPhase.CQ,
        trigger_time + tq_to_cq_dur: DisruptionPhase.CQ,
    }
    return disrupt_phase_dict


def generate_island_rotation_phase_trajectory(trigger_time: float, rot_dur: float, locking_dur: float):
    # TODO(allenw): we want a rectilinear interpolation scheme.
    rot_phase_dict = {
        0.0: IslandRotationPhase.NONE,
        trigger_time - dt: IslandRotationPhase.NONE,
        trigger_time: IslandRotationPhase.ROTATING,
        trigger_time + rot_dur - dt: IslandRotationPhase.ROTATING,
        trigger_time + rot_dur: IslandRotationPhase.DECELERATING,
        trigger_time + rot_dur + locking_dur - dt: IslandRotationPhase.DECELERATING,
        trigger_time + rot_dur + locking_dur: IslandRotationPhase.LOCKED,
    }
    return rot_phase_dict


initial_state = Tearing.State(W=0.0, F=0.0)


rot_dur = 1.0
trigger_time = 5.0
disrupt_time = 6.5
dur_tq_to_spike = 1e-3
dur_cq = 10e-3
survival_time = 0.3
locking_dur = 0.2

params = Tearing.Params(
    q2_rot_freq=7e3,  # Hz
    rot_dur=1.0,  # s
    locking_dur=locking_dur,  # s
    disruption_phase=generate_disruption_phase_trajectory(disrupt_time, dur_tq_to_spike),
    island_rotation_phase=generate_island_rotation_phase_trajectory(trigger_time, rot_dur, locking_dur),
)

In [ ]:
tearing_module = Tearing(config=config)

sol_xarray = simulate(tearing_module, time_base, initial_state, params)

In [ ]:
from popsim.visualize import visualize_time_series

visualize_time_series(sol_xarray)